In [1]:
# Scenario: Customer Support Chatbot Workflow
# Imagine a company wants to build a chatbot that helps customers with quick answers. The workflow is modeled as a graph of states:

# - State Definition (BotState)
# - The chatbot keeps track of:
# - The question asked by the customer.
# - The answer generated.
# - The history of all past questions.
# - Think of this as the chatbot’s notebook where it records the conversation.

# - Nodes (Functions)
# - get_answer:
# When a customer asks, “What are your store hours?”, the chatbot looks at the question and generates a placeholder answer like “Answer to: What are your store hours?”.
# It also adds the question to the history log.
# - format_output:
# Before sending the response back to the customer, the chatbot reformats it into a friendly style:
# “Bot says: Answer to: What are your store hours?”

# - Graph Flow
# - The chatbot starts at the get_answer node (entry point).
# - Once the answer is generated, it flows to the format_output node.
# - Finally, the conversation ends at END, meaning the chatbot has
#  delivered its response.


from langgraph.graph import StateGraph, END
from typing import TypedDict

# 1. Define State
class BotState(TypedDict):
    question: str
    answer: str
    history: list

# 2. Define Nodes (functions)
def get_answer(state: BotState):
    q = state["question"]
    # In real app: call LLM here
    ans = f"Answer to: {q}"
    return {"answer": ans,
            "history": state["history"] + [q]}

def format_output(state: BotState):
    return {"answer": f"Bot says: {state['answer']}"}

# 3. Build the Graph
graph = StateGraph(BotState)
graph.add_node("get_answer", get_answer)
graph.add_node("format", format_output)

# 4. Add Edges
graph.set_entry_point("get_answer")
graph.add_edge("get_answer", "format")
graph.add_edge("format", END)


In [3]:
# Scenario: Customer Support Chatbot (Question-Based)
# Imagine a company has deployed a chatbot that answers customer
#  questions by calling the Groq API. The workflow is modeled as a
#  graph of states, where each customer query flows through nodes until
#   a response is delivered.

# 1. State Definition
# The chatbot maintains a notebook-like state:
# - question → The customer’s query.
# - answer → The response generated by Groq.
# - history → A log of all past questions.


from langgraph.graph import StateGraph, END
from typing import TypedDict
import requests
from google.colab import userdata

# 1. Define State
class BotState(TypedDict):
    question: str
    answer: str
    history: list

# 2. Define Nodes (functions)
def get_answer(state: BotState):
    q = state["question"]
    groq_api_key = userdata.get('groq_api_key')

    if not groq_api_key:
        raise ValueError("Groq API key not found in Colab secrets. Please set 'Groq_api'.")

    # Call Groq API
    response = requests.post(
        "https://api.groq.com/openai/v1/chat/completions",
        headers={"Authorization": f"Bearer {groq_api_key}"},
        json={
            # IMPORTANT: The list of supported models by Groq API changes frequently.
            # Please refer to the official Groq documentation (https://console.groq.com/docs/models)
            # to find a currently active and supported model name and replace 'YOUR_GROQ_MODEL_NAME_HERE' below.
            # Examples of often available models include 'llama3-8b-8192' or 'llama3-70b-8192',
            # but these can also become decommissioned.
            "model": "llama-3.1-8b-instant",   # Changed to a currently active model
            "messages": [{"role": "user", "content": q}],
        }
    )

    if response.status_code != 200:
        try:
            error_details = response.json()
        except requests.exceptions.JSONDecodeError:
            error_details = response.text
        raise Exception(f"Groq API error (Status: {response.status_code}): {error_details}")

    response_json = response.json()
    if "choices" not in response_json or not response_json["choices"]:
        raise ValueError(f"Unexpected API response format: 'choices' key missing or empty. Full response: {response_json}")

    # Extract answer from Groq response
    ans = response_json["choices"][0]["message"]["content"]

    return {
        "answer": ans,
        "history": state["history"] + [q]
    }

def format_output(state: BotState):
    return {"answer": f"Bot says: {state['answer']}"}

# 3. Build the Graph
graph = StateGraph(BotState)
graph.add_node("get_answer", get_answer)
graph.add_node("format", format_output)

# 4. Add Edges
graph.set_entry_point("get_answer")
graph.add_edge("get_answer", "format")
graph.add_edge("format", END)

# 5. Example Run
if __name__ == "__main__":
    # Initial state
    state = {"question": "What are your store hours?", "answer": "", "history": []}

    # Run the graph
    app = graph.compile()
    result = app.invoke(state)

    print(result["answer"])

Bot says: I'm a large language model, I don't have a physical store. I exist solely as a digital entity, and I'm available 24/7 to assist with your queries. You can access me through this chat platform or other interfaces at any time.


In [4]:
# Scenario: AI-Powered Study Assistant (Flashcard-Based)

# 1. State Definition
# The assistant maintains a notebook-like state for each learner:
# - topic → The subject the learner is studying (e.g., "Photosynthesis").
# - flashcard → A generated Q&A card created by Groq (question on one side, answer on the other).
# - progress → A log of all past flashcards attempted, including whether the learner got them correct or not.

# 2. Workflow (Graph of States)
# Each learner interaction flows through nodes until a flashcard is delivered:

# - Input Node
# - Learner provides a topic or asks for practice (e.g., "Test me on cell biology").
# - State updates: topic = "cell biology"

# - Generation Node (Groq API)
# - Groq generates a flashcard:
# - flashcard.question = "What organelle is known as the powerhouse of the cell?"
# - flashcard.answer = "Mitochondria"

# - Response Node
# - Assistant presents the flashcard question to the learner.

# - Evaluation Node
# - Learner responds with their answer.
# - Assistant checks correctness and updates progress.

# - History Node
# - Logs the flashcard attempt:
# - progress = [{question, learner_answer, correct/incorrect}]

from langgraph.graph import StateGraph, END
from typing import TypedDict, List, Dict
import requests
from google.colab import userdata

# 1. State Definition
class StudyState(TypedDict):
    topic: str
    flashcard: Dict
    user_answer: str
    feedback: str
    progress: List[Dict]

# 2. Node: Generate Flashcard (Groq API)
def generate_flashcard(state: StudyState):
    topic = state["topic"]
    api_key = userdata.get("groq_api_key")

    prompt = f"Create a simple flashcard for the topic '{topic}'. Return in format:\nQuestion: ...\nAnswer: ..."

    response = requests.post(
        "https://api.groq.com/openai/v1/chat/completions",
        headers={"Authorization": f"Bearer {api_key}"},
        json={
            "model": "llama-3.1-8b-instant",
            "messages": [{"role": "user", "content": prompt}]
        }
    )

    data = response.json()
    content = data["choices"][0]["message"]["content"]

    # Parse response
    lines = content.split("\n")
    question = lines[0].replace("Question:", "").strip()
    answer = lines[1].replace("Answer:", "").strip()

    return {
        "flashcard": {"question": question, "answer": answer}
    }

# 3. Node: Ask Question
def ask_question(state: StudyState):
    print("\n📘 Question:", state["flashcard"]["question"])
    user_ans = input("Your Answer: ")

    return {"user_answer": user_ans}

# 4. Node: Evaluate Answer
def evaluate_answer(state: StudyState):
    correct_answer = state["flashcard"]["answer"].lower()
    user_answer = state["user_answer"].lower()

    if correct_answer in user_answer:
        feedback = "✅ Correct!"
        result = "correct"
    else:
        feedback = f"❌ Incorrect! Correct answer: {correct_answer}"
        result = "incorrect"

    return {
        "feedback": feedback,
        "progress": state["progress"] + [{
            "question": state["flashcard"]["question"],
            "your_answer": state["user_answer"],
            "result": result
        }]
    }

# 5. Node: Show Feedback
def show_feedback(state: StudyState):
    print(state["feedback"])
    return {}

# 6. Build Graph
graph = StateGraph(StudyState)

graph.add_node("generate", generate_flashcard)
graph.add_node("ask", ask_question)
graph.add_node("evaluate", evaluate_answer)
graph.add_node("feedback", show_feedback)

graph.set_entry_point("generate")

graph.add_edge("generate", "ask")
graph.add_edge("ask", "evaluate")
graph.add_edge("evaluate", "feedback")
graph.add_edge("feedback", END)

# 7. Run Example
if __name__ == "__main__":
    app = graph.compile()

    state = {
        "topic": "cell biology",
        "flashcard": {},
        "user_answer": "",
        "feedback": "",
        "progress": []
    }

    result = app.invoke(state)

    print("\n📊 Progress:", result["progress"])


📘 Question: Here's a simple flashcard for the topic 'cell biology':
Your Answer: Test me on photosynthesis
✅ Correct!

📊 Progress: [{'question': "Here's a simple flashcard for the topic 'cell biology':", 'your_answer': 'Test me on photosynthesis', 'result': 'correct'}]


In [6]:
# Scenario: AI-Powered Project Tracker (Task-Based)

# 1. State Definition
# The assistant maintains a notebook-like state for each project:
# - task → The specific work item or milestone (e.g., "Prepare Q1 financial report").
# - status → The current state of the task (e.g., "in progress", "completed", "blocked").
# - log → A history of all updates, including who made them and when.

# 2. Workflow (Graph of States)
# Each project update flows through nodes until the task status is refreshed:

# - Input Node
# - Team member submits an update (e.g., "Report draft completed").
# - State updates: task = "Q1 financial report"

# - Processing Node (Groq API)
# - Groq interprets the update and assigns a status:
# - status = "completed"

# - Response Node
# - Assistant confirms the update back to the team:
# - "Task Q1 financial report marked as completed."

# - History Node
# - Logs the update:
# - log = [{task: "Q1 financial report", update: "draft completed", status: "completed", timestamp}]

from langgraph.graph import StateGraph, END
from typing import TypedDict, List, Dict
import requests
import re
from google.colab import userdata

# 1. State Definition
class StudyState(TypedDict):
    topic: str
    flashcard: Dict
    user_answer: str
    feedback: str
    progress: List[Dict]

# 2. Helper: Robust Parser
def parse_flashcard(text: str):
    # Try strict format first
    match = re.search(r"Question:\s*(.*?)\nAnswer:\s*(.*)", text, re.DOTALL)

    if match:
        return {
            "question": match.group(1).strip(),
            "answer": match.group(2).strip()
        }

    # Fallback: extract first Q/A if multiple exist
    questions = re.findall(r"Question.*?:\s*(.*)", text)
    answers = re.findall(r"Answer.*?:\s*(.*)", text)

    if questions and answers:
        return {
            "question": questions[0].strip(),
            "answer": answers[0].strip()
        }

    # अंतिम fallback
    return {
        "question": "Unable to parse question",
        "answer": "Unable to parse answer"
    }

# 3. Node: Generate Flashcard
def generate_flashcard(state: StudyState):
    topic = state["topic"]
    api_key = userdata.get("groq_api_key")

    if not api_key:
        raise ValueError("Groq API key missing")

    prompt = f"""
Generate ONLY ONE flashcard for the topic '{topic}'.

Strict format:
Question: <your question>
Answer: <your answer>

Rules:
- Do NOT generate multiple questions
- Do NOT add extra text
"""

    response = requests.post(
        "https://api.groq.com/openai/v1/chat/completions",
        headers={"Authorization": f"Bearer {api_key}"},
        json={
            "model": "llama-3.1-8b-instant",
            "messages": [{"role": "user", "content": prompt}]
        },
        timeout=10
    )

    if response.status_code != 200:
        raise Exception(f"API Error: {response.text}")

    content = response.json()["choices"][0]["message"]["content"]

    flashcard = parse_flashcard(content)

    return {"flashcard": flashcard}

# 4. Node: Ask Question
def ask_question(state: StudyState):
    print("\n📘 Question:", state["flashcard"]["question"])
    user_ans = input("✍️ Your Answer: ")
    return {"user_answer": user_ans}

# 5. Node: Evaluate Answer
def evaluate_answer(state: StudyState):
    correct_answer = state["flashcard"]["answer"].lower().strip()
    user_answer = state["user_answer"].lower().strip()

    # Simple matching (can be improved later)
    if correct_answer in user_answer or user_answer in correct_answer:
        feedback = "✅ Correct!"
        result = "correct"
    else:
        feedback = f"❌ Incorrect! Correct answer: {correct_answer}"
        result = "incorrect"

    return {
        "feedback": feedback,
        "progress": state["progress"] + [{
            "question": state["flashcard"]["question"],
            "correct_answer": correct_answer,
            "your_answer": user_answer,
            "result": result
        }]
    }

# 6. Node: Show Feedback
def show_feedback(state: StudyState):
    print(state["feedback"])
    return {}

# 7. Build Graph
graph = StateGraph(StudyState)

graph.add_node("generate", generate_flashcard)
graph.add_node("ask", ask_question)
graph.add_node("evaluate", evaluate_answer)
graph.add_node("feedback", show_feedback)

graph.set_entry_point("generate")

graph.add_edge("generate", "ask")
graph.add_edge("ask", "evaluate")
graph.add_edge("evaluate", "feedback")
graph.add_edge("feedback", END)

# 8. Run Loop (Multi-turn 🔥)
if __name__ == "__main__":
    app = graph.compile()

    state = {
        "topic": "",
        "flashcard": {},
        "user_answer": "",
        "feedback": "",
        "progress": []
    }

    print("📚 AI Study Assistant Started (type 'exit' to quit)\n")

    while True:
        topic = input("\n📌 Enter topic: ")

        if topic.lower() == "exit":
            break

        state["topic"] = topic

        result = app.invoke(state)

        state["progress"] = result["progress"]

        print("\n📊 Progress so far:")
        for i, p in enumerate(state["progress"], 1):
            print(f"{i}. {p['question']} → {p['result']}")

    print("\n👋 Session Ended")

📚 AI Study Assistant Started (type 'exit' to quit)


📌 Enter topic: cell biology

📘 Question: What is the primary function of the mitochondria in a cell?
✍️ Your Answer: 📘 Question: What is the powerhouse of the cell? ✍️ Your Answer:
❌ Incorrect! Correct answer: the primary function of the mitochondria is to generate energy for the cell through the process of cellular respiration.

📊 Progress so far:
1. What is the primary function of the mitochondria in a cell? → incorrect

📌 Enter topic: cell biology

📘 Question: What is the primary function of the cell membrane in eukaryotic cells?
✍️ Your Answer: mitochondria
❌ Incorrect! Correct answer: the primary function of the cell membrane in eukaryotic cells is to regulate the movement of substances in and out of the cell by controlling diffusion and active transport.

📊 Progress so far:
1. What is the primary function of the mitochondria in a cell? → incorrect
2. What is the primary function of the cell membrane in eukaryotic cells? → incor

In [7]:
# Scenario: Customer Support Call Center
# A company runs a support chatbot that needs to route customer queries to the right department. Instead of one big script, they design a state graph where each node represents a specialized agent.

# 1. State Definition (SupportState)
# The chatbot keeps track of:
# - query → What the customer asked.
# - category → Which department it belongs to (billing, tech, general).
# - response → What the bot replies with.
# Think of this as the customer’s “ticket form.”

# 2. Router Node (route_query)
# When a customer types a question, the router scans it:
# - If the query mentions “bill” or “payment”, it routes to billing_agent.
# - If it mentions “error” or “bug”, it routes to tech_agent.
# - Otherwise, it defaults to general_agent.
# This is like a receptionist deciding which desk you should go to.

# 3. Agent Nodes
# - billing_agent → Replies with “Billing dept: [query]”.
# - tech_agent → Replies with “Tech support: [query]”.
# - general_agent → Replies with “General help: [query]”.
# Each agent specializes in its own type of problem.

# 4. Graph Flow
# - Entry point: router.
# - Router decides the path based on the query.
# - The query flows into the correct agent node.
# - The agent generates a response and ends the conversation.


from langgraph.graph import StateGraph, END
from typing import TypedDict, Literal

class SupportState(TypedDict):
    query: str
    category: str   # "billing" | "tech" | "general"
    response: str

# Router: reads state, returns next node name
def route_query(state: SupportState) -> str:
    q = state["query"].lower()
    if "bill" in q or "payment" in q:
        return "billing_agent"
    elif "error" in q or "bug" in q:
        return "tech_agent"
    else:
        return "general_agent"

def billing_agent(state):
    return {"response": "Billing dept: " + state["query"]}

def tech_agent(state):
    return {"response": "Tech support: " + state["query"]}

def general_agent(state):
    return {"response": "General help: " + state["query"]}

# Build graph with conditional routing
g = StateGraph(SupportState)
g.add_node("billing_agent", billing_agent)
g.add_node("tech_agent", tech_agent)
g.add_node("general_agent", general_agent)

# One entry point routes to 3 different nodes!
g.set_entry_point("router")
g.add_conditional_edges(
    "router",    # from node
    route_query, # function that returns next node
    {            # mapping: return value → node name
        "billing_agent":  "billing_agent",
        "tech_agent":     "tech_agent",
        "general_agent":  "general_agent",
    }
)


In [8]:
from langgraph.graph import StateGraph, END
from typing import TypedDict

class ResearchState(TypedDict):
    topic: str
    search_results: list
    analysis: str
    summary: str
    steps_done: int

# Node 1: Search for information
def search_web(state: ResearchState):
    print(f"\u001f Searching: {state['topic']}")
    # Simulate web search results
    new_results = [
        f"Article 1 about {state['topic']}",
        f"Article 2 about {state['topic']}",
    ]
    return {"search_results": state["search_results"] + new_results,
            "steps_done": state["steps_done"] + 1}

# Node 2: Analyze the results
def analyze_results(state: ResearchState):
    print(f"\u001f Analyzing {len(state['search_results'])} results")
    analysis = f"Key insights from {len(state['search_results'])} sources"
    return {"analysis": analysis,
            "steps_done": state["steps_done"] + 1}

# Node 3: Summarize
def summarize(state: ResearchState):
    print("\u001f\u001f Generating summary...")
    summary = f"Summary: {state['analysis']}"
    return {"summary": summary}

# Node 4: Check if we need more research
def should_continue(state: ResearchState) -> str:
    if len(state["search_results"]) < 3:
        return "search_web"   # Loop back!
    return END                 # Done

# Build the graph
g = StateGraph(ResearchState)
g.add_node("search_web",  search_web)
g.add_node("analyze",     analyze_results)
g.add_node("summarize",   summarize)

g.set_entry_point("search_web")
g.add_edge("search_web", "analyze")
g.add_conditional_edges("analyze", should_continue,
    {"search_web": "search_web", END: "summarize"})
g.add_edge("summarize", END)

app = g.compile()
result = app.invoke(
    {
        "topic": "Quantum Computing",
        "search_results": [],
        "analysis": "",
        "summary": "",
        "steps_done": 0
    }
)
print(result["summary"])

 Searching: Quantum Computing
 Analyzing 2 results
 Searching: Quantum Computing
 Analyzing 4 results
 Generating summary...
Summary: Key insights from 4 sources


In [9]:
from langgraph.graph import StateGraph, END
from typing import TypedDict
import requests
from google.colab import userdata

# 1. Define State
class ResearchState(TypedDict):
    topic: str
    search_results: list
    analysis: str
    summary: str
    steps_done: int

# 2. Helper: Groq API call
def groq_call(prompt: str, model: str = "llama-3.1-8b-instant"): # Changed model to a known working one
    groq_api_key = userdata.get('groq_api_key')

    if not groq_api_key:
        raise ValueError("Groq API key not found in Colab secrets. Please set 'Groq_api'.")

    response = requests.post(
        "https://api.groq.com/openai/v1/chat/completions",
        headers={"Authorization": f"Bearer {groq_api_key}"},
        json={
            "model": model,
            "messages": [{"role": "user", "content": prompt}],
        }
    )

    if response.status_code != 200:
        try:
            error_details = response.json()
        except requests.exceptions.JSONDecodeError:
            error_details = response.text
        raise Exception(f"Groq API error (Status: {response.status_code}): {error_details}")

    response_json = response.json()
    if "choices" not in response_json or not response_json["choices"]:
        raise ValueError(f"Unexpected API response format: 'choices' key missing or empty. Full response: {response_json}")

    return response_json["choices"][0]["message"]["content"]

# 3. Nodes
def search_web(state: ResearchState):
    print(f"🔍 Searching: {state['topic']}")
    # Call Groq to generate snippets
    new_results = [
        groq_call(f"Give me a short article snippet about {state['topic']}"),
        groq_call(f"Give me another snippet about {state['topic']}")
    ]
    results = state["search_results"] + new_results
    return {
        "search_results": results,
        "steps_done": state["steps_done"] + 1
    }

def analyze_results(state: ResearchState):
    print(f"📊 Analyzing {len(state['search_results'])} results")
    joined_results = "\n".join(state["search_results"])
    analysis = groq_call(f"Analyze these sources and extract key insights:\n{joined_results}")
    return {
        "analysis": analysis,
        "steps_done": state["steps_done"] + 1
    }

def summarize(state: ResearchState):
    print("✍️ Generating summary...")
    summary = groq_call(f"Summarize this analysis in simple terms:\n{state['analysis']}")
    return {"summary": summary}

def should_continue(state: ResearchState) -> str:
    if len(state["search_results"]) < 3:
        return "search_web"   # Loop back until enough results
    return "summarize"        # Once we have 3+, move to summary

# 4. Build the graph
g = StateGraph(ResearchState)
g.add_node("search_web",  search_web)
g.add_node("analyze",     analyze_results)
g.add_node("summarize",   summarize)

g.set_entry_point("search_web")
g.add_edge("search_web", "analyze")
g.add_conditional_edges("analyze", should_continue,
    {"search_web": "search_web", "summarize": "summarize"})
g.add_edge("summarize", END)

# 5. Run the graph
if __name__ == "__main__":
    app = g.compile()
    result = app.invoke({
        "topic": "Quantum Computing",
        "search_results": [], "analysis": "",
        "summary": "", "steps_done": 0
    })
    print("\n✅ Final Summary:\n", result["summary"])

🔍 Searching: Quantum Computing
📊 Analyzing 2 results
🔍 Searching: Quantum Computing
📊 Analyzing 4 results
✍️ Generating summary...

✅ Final Summary:
 **Summary of the Analysis:**

Quantum computing is a new and powerful technology that uses tiny units called qubits to process information much faster than regular computers. It has the potential to solve complex problems in various fields such as:

- **Materials Science**: Creating new materials with unique properties.
- **Chemical Reactions**: Studying and optimizing chemical reactions.
- **Biology**: Understanding complex biological systems like protein folding.
- **Logistics**: Improving supply chain management and transportation.
- **Finance**: Developing new financial models and algorithms.
- **Medicine**: Finding new treatments and understanding diseases.
- **Energy Storage**: Improving battery performance and efficiency.

**Current Developments:**

Many companies and research organizations are working on building quantum computers

In [11]:
# Scenario: AI Symptom Tracker (Question-Based)

# 1. State Definition
# The assistant maintains a notebook-like state for each patient:
# - symptom → The patient’s reported issue (e.g., "persistent cough").
# - observations → Notes or snippets generated by Groq about possible causes or related conditions.
# - analysis → A synthesized interpretation of the observations.
# - recommendation → A simplified, non-medical summary suggesting next steps (e.g., "consult a doctor if symptoms persist").
# - steps_done → A counter tracking progress through the workflow.

# 2. Workflow (Graph of States)
# Each patient query flows through nodes:

# - Symptom Input Node
# - Patient reports a symptom.
# - State updates: symptom = "persistent cough"

# - Observation Node (Groq API)
# - Groq generates possible related factors or general information.
# - Updates observations.

# - Analysis Node
# - Joins observations and extracts key insights.
# - Updates analysis.

# - Conditional Node
# - If fewer than 3 observations are collected → loop back to Observation Node.
# - If 3+ observations are available → move to Recommendation Node.

# - Recommendation Node
# - Generates a simplified, non-medical recommendation (e.g., "Seek medical advice if cough lasts more than 2 weeks").
# - Updates recommendation.

# - End Node
# - Delivers the final recommendation to the patient.

from langgraph.graph import StateGraph, END
from typing import TypedDict, List
import requests
from google.colab import userdata

# 1. State Definition
class HealthState(TypedDict):
    symptom: str
    observations: List[str]
    analysis: str
    recommendation: str
    steps_done: int

# 2. Node: Observation (Groq API)
def observation_node(state: HealthState):
    api_key = userdata.get("groq_api_key")

    prompt = f"""
Patient symptom: {state['symptom']}

Provide ONE general observation or possible cause (non-diagnostic).
Keep it simple and informative.
"""

    response = requests.post(
        "https://api.groq.com/openai/v1/chat/completions",
        headers={"Authorization": f"Bearer {api_key}"},
        json={
            "model": "llama-3.1-8b-instant",
            "messages": [{"role": "user", "content": prompt}]
        },
        timeout=10
    )

    content = response.json()["choices"][0]["message"]["content"]

    return {
        "observations": state["observations"] + [content],
        "steps_done": state["steps_done"] + 1
    }

# 3. Node: Analysis
def analysis_node(state: HealthState):
    combined = " ".join(state["observations"])

    analysis = f"Based on observations: {combined[:200]}..."

    return {"analysis": analysis}

# 4. Conditional Node
def check_condition(state: HealthState):
    if len(state["observations"]) < 3:
        return "observe_again"
    else:
        return "analyze"

# 5. Node: Recommendation
def recommendation_node(state: HealthState):
    api_key = userdata.get("groq_api_key")

    prompt = f"""
Symptom: {state['symptom']}
Observations: {state['observations']}

Give a simple, safe, non-medical recommendation.
Avoid diagnosis.
"""

    response = requests.post(
        "https://api.groq.com/openai/v1/chat/completions",
        headers={"Authorization": f"Bearer {api_key}"},
        json={
            "model": "llama-3.1-8b-instant",
            "messages": [{"role": "user", "content": prompt}]
        },
        timeout=10
    )

    content = response.json()["choices"][0]["message"]["content"]

    return {"recommendation": content}

# 6. Node: Final Output
def final_node(state: HealthState):
    print("\n🩺 Symptom:", state["symptom"])

    print("\n📌 Observations:")
    for i, obs in enumerate(state["observations"], 1):
        print(f"{i}. {obs}")

    print("\n🧠 Analysis:")
    print(state["analysis"])

    print("\n💡 Recommendation:")
    print(state["recommendation"])

    return {}

# 7. Build Graph
graph = StateGraph(HealthState)

graph.add_node("observe", observation_node)
graph.add_node("analyze", analysis_node)
graph.add_node("recommend", recommendation_node)
graph.add_node("final", final_node)

graph.set_entry_point("observe")

# Conditional loop
graph.add_conditional_edges(
    "observe",
    check_condition,
    {
        "observe_again": "observe",
        "analyze": "analyze"
    }
)

graph.add_edge("analyze", "recommend")
graph.add_edge("recommend", "final")
graph.add_edge("final", END)

# 8. Run Example
if __name__ == "__main__":
    app = graph.compile()

    symptom = input("🩺 Enter your symptom: ")

    state = {
        "symptom": symptom,
        "observations": [],
        "analysis": "",
        "recommendation": "",
        "steps_done": 0
    }

    app.invoke(state)

🩺 Enter your symptom: headache and dizziness

🩺 Symptom: headache and dizziness

📌 Observations:
1. Possible cause: Dehydration.

Dehydration can lead to headaches and dizziness due to the body's lack of essential fluids and electrolytes, which are necessary for proper neurological function and blood circulation.
2. Possible cause: Dehydration may be contributing to the patient's headache and dizziness.
3. A possible cause for headaches and dizziness could be dehydration.

🧠 Analysis:
Based on observations: Possible cause: Dehydration.

Dehydration can lead to headaches and dizziness due to the body's lack of essential fluids and electrolytes, which are necessary for proper neurological function and bloo...

💡 Recommendation:
Based on the symptoms and observations, a simple and safe non-medical recommendation is to:

- Drink plenty of water: Try to consume at least 8-10 glasses of water throughout the day to replenish lost fluids.
- Stay hydrated: Eat hydrating foods like fruits, salad

In [13]:
# Scenario: AI-Assisted Email Workflow (Question-Based)
# Context
# A company deploys an AI-powered email assistant to help employees draft, review, and send professional emails.
# The workflow is modeled as a graph of states, where each email task flows through nodes until it is either approved
# and sent or rejected.

# 1. State Definition
# The assistant maintains a notebook-like state:
# - task → The subject or purpose of the email (e.g., "Q3 Report").
# - draft → The AI-generated email draft.
# - approved → A flag indicating whether the human reviewer has approved the draft.

# 2. Workflow (Graph of States)
# Each email task flows through nodes:
# - Draft Node
# - AI generates a draft email based on the task.
# - Updates draft.
# - Review Node (Interrupt)
# - Execution pauses here.
# - Human reviewer inspects the draft and decides whether to approve or reject.
# - Updates approved.
# - Send Node
# - If approved = True → Email is sent.
# - If approved = False → Email is rejected.
# - Updates task with final status (SENT or REJECTED).
# - End Node
# - Workflow completes.

# 3. Example Flow
# - Employee: "Draft an email for the Q3 Report."
# - State: task = "Q3 Report"
# - Assistant drafts:
# Dear User,
# Regarding: Q3 Report
# [AI drafted content]
# - Human reviews → Approves draft (approved = True)
# - Assistant sends → task = "SENT: Q3 Report"
# - Final Output: ✅ Email delivered.

# 👉 Scenario Question:
# "Imagine you are designing an AI-powered email assistant that drafts emails, pauses for human review, and
# then either sends or rejects them. How would you structure the state and workflow graph to ensure accountability
#  and human oversight in the process?"

from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
from typing import TypedDict
import requests
from google.colab import userdata

# 1. Define State
class EmailState(TypedDict):
    task: str
    draft: str
    approved: bool

# 2. Helper: Groq API call
def groq_call(prompt: str, model: str = "llama-3.1-8b-instant"):
    groq_api_key = userdata.get('groq_api_key')
    if not groq_api_key:
        raise ValueError("Groq API key not found in Colab secrets. Please set 'Groq_api'.")

    response = requests.post(
        "https://api.groq.com/openai/v1/chat/completions",
        headers={"Authorization": f"Bearer {groq_api_key}"},
        json={
            "model": model,
            "messages": [{"role": "user", "content": prompt}],
        }
    )

    if response.status_code != 200:
        try:
            error_details = response.json()
        except requests.exceptions.JSONDecodeError:
            error_details = response.text
        raise Exception(f"Groq API error (Status: {response.status_code}): {error_details}")

    response_json = response.json()
    if "choices" not in response_json or not response_json["choices"]:
        raise ValueError(f"Unexpected API response format: {response_json}")

    return response_json["choices"][0]["message"]["content"]

# 3. Nodes
def draft_email(state: EmailState):
    print(f"📝 Drafting email for task: {state['task']}")
    draft = groq_call(f"Draft a professional email regarding: {state['task']}")
    return {"draft": draft}

def human_review(state: EmailState):
    # Interrupt node: waits for human approval
    print(f"📧 Draft ready for review:\n\n{state['draft']}\n")
    return {}  # Pauses here until human updates 'approved'

def send_email(state: EmailState):
    if state.get("approved", False):
        print("✅ Email approved and sent.")
        return {"task": f"SENT: {state['task']}"}
    else:
        print("❌ Email rejected.")
        return {"task": f"REJECTED: {state['task']}"}

# 4. Build Graph
g = StateGraph(EmailState)
g.add_node("draft", draft_email)
g.add_node("review", human_review)
g.add_node("send", send_email)

g.set_entry_point("draft")
g.add_edge("draft", "review")
g.add_edge("review", "send")
g.add_edge("send", END)

# 5. Checkpointer
checkpointer = MemorySaver()
app = g.compile(
    checkpointer=checkpointer,
    interrupt_before=["review"]  # Pause before review
)

# 6. Run Workflow
thread = {"configurable": {"thread_id": "email-1"}}

# Step 1: Draft email
app.invoke({"task": "Q3 Report", "draft": "", "approved": False}, thread)

# Step 2: Human reviews draft and resumes
app.invoke({"approved": True}, thread)

📝 Drafting email for task: Q3 Report
📝 Drafting email for task: Q3 Report


{'task': 'Q3 Report',
 'draft': "Here's a sample email regarding the Q3 report:\n\nSubject: Q3 Report - Key Highlights and Insights\n\nDear [Recipient's Name],\n\nI am writing to share the Q3 report, which provides an overview of our company's performance during the third quarter of the year. The report highlights our achievements, areas for improvement, and key insights gained during this period.\n\nBelow are the key highlights from the Q3 report:\n\n- Revenue growth: We have seen a significant increase in revenue compared to the same period last year, with a growth rate of [X]%.\n- Operational efficiency: Our operational efficiency has improved, resulting in cost savings of [X]%.\n- Key milestones achieved: We have successfully completed [project/initiative name] and have made significant progress on [project/initiative name].\n\nSome of the key insights from the report are:\n\n- The market trends and competitor analysis indicate a shift in consumer behavior, with a growing demand fo

In [16]:
# Scenario Question
# "Imagine you are designing an AI-powered assistant that drafts structured reports, pauses for human review, and then either publishes or rejects them. How would you structure the state and workflow graph to ensure accountability and human oversight in the process?"

from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
from typing import TypedDict
import requests
from google.colab import userdata

# 1. State Definition
class ReportState(TypedDict):
    task: str
    draft: str
    approved: bool
    reviewer_comments: str
    status: str   # PENDING / PUBLISHED / REJECTED

# 2. Groq API Helper
def groq_call(prompt: str, model: str = "llama-3.1-8b-instant"):
    api_key = userdata.get("groq_api_key")
    if not api_key:
        raise ValueError("Groq API key missing")

    response = requests.post(
        "https://api.groq.com/openai/v1/chat/completions",
        headers={"Authorization": f"Bearer {api_key}"},
        json={
            "model": model,
            "messages": [{"role": "user", "content": prompt}],
        },
        timeout=10
    )

    if response.status_code != 200:
        raise Exception(f"API Error: {response.text}")

    return response.json()["choices"][0]["message"]["content"]

# 3. Node: Draft Report
def draft_report(state: ReportState):
    print(f"\n📝 Generating report for: {state['task']}")

    prompt = f"""
Create a structured professional report on: {state['task']}

Include:
- Title
- Introduction
- Key Points
- Conclusion
"""

    draft = groq_call(prompt)

    return {
        "draft": draft,
        "status": "PENDING"
    }

# 4. Node: Human Review (Interrupt)
def review_node(state: ReportState):
    print("\n📄 Draft Report:\n")
    print(state["draft"])
    print("\n⏸ Waiting for human review...")
    return {}

# 5. Node: Decision
def decision_node(state: ReportState):
    if state.get("approved", False):
        print("\n✅ Report approved and published.")
        return {"status": f"PUBLISHED: {state['task']}"}
    else:
        print("\n❌ Report rejected.")
        return {"status": f"REJECTED: {state['task']}"}

# 6. Build Graph
graph = StateGraph(ReportState)

graph.add_node("draft", draft_report)
graph.add_node("review", review_node)
graph.add_node("decision", decision_node)

graph.set_entry_point("draft")

graph.add_edge("draft", "review")
graph.add_edge("review", "decision")
graph.add_edge("decision", END)

# 7. Checkpointer
checkpointer = MemorySaver()

app = graph.compile(
    checkpointer=checkpointer,
    interrupt_before=["review"]   # pause before review
)

# 8. Run Workflow
if __name__ == "__main__":
    thread = {"configurable": {"thread_id": "report-1"}}

    # Step 1: Generate Draft (runs till interrupt)
    app.invoke(
        {
            "task": "AI in Healthcare",
            "draft": "",
            "approved": False,
            "reviewer_comments": "",
            "status": ""
        },
        thread
    )

    # Step 2: Human Review
    print("\n👩‍💼 Review Step:")
    decision = input("Approve report? (yes/no): ").lower()
    comments = input("Reviewer comments: ")

    approved = True if decision == "yes" else False

    # ✅ Step 3: Update state (IMPORTANT)
    app.update_state(
        thread,
        {
            "approved": approved,
            "reviewer_comments": comments
        }
    )

    # ✅ Step 4: Resume from checkpoint (IMPORTANT)
    result = app.invoke(None, thread)

    print("\n📊 Final Status:", result["status"])


📝 Generating report for: AI in Healthcare

👩‍💼 Review Step:
Approve report? (yes/no): yes
Reviewer comments: Well-structured report. Content is clear and professional. Approved for publishing.

📄 Draft Report:

**Title:** "The Integration of Artificial Intelligence in Healthcare: Transforming Patient Care and Outcomes"

**Introduction:**

The healthcare industry has undergone significant transformations with the advent of technology, and artificial intelligence (AI) is revolutionizing the way patient care is delivered. AI in healthcare has the potential to improve diagnosis accuracy, streamline clinical workflows, enhance patient engagement, and enable more effective disease management. In this report, we will explore the key points of AI in healthcare, highlighting its benefits, applications, and future prospects.

**Key Points:**

1. **Diagnostic Accuracy:**
	* AI algorithms can analyze vast amounts of medical data, such as images and lab results, to improve diagnosis accuracy and r